
# InterLeaf — RRCP Sampling Record and Sample Preparation

This Colab notebook documents and validates the **frozen 250-post RRCP evaluation sample** used in the InterLeaf paper and prepares the input consumed by the prediction notebook.

The study sample contains:

- **200 naturalistic core posts**
- **50 safety-enriched challenge posts**
  - 20 direct self-harm/suicidality candidates
  - 10 indirect despair candidates
  - 10 medication/medical-risk candidates
  - 10 violence/abuse candidates
- random seed: **20260903**
- exactly one sampled post per Reddit author
- no duplicate source IDs or normalized texts

The authoritative sampling record is the researcher-only workbook `reddit_250_sampling.xlsx`.

> **Important:** This notebook does not redistribute or re-scrape the full RRCP corpus. It validates the already-frozen sample used in the study and creates the exact two-column input required by `Interleaf_AI_Pipeline_Predictions_Reddit.ipynb`. Re-running the initial draw from all 86,537 RRCP posts would require the original RRCP source table and the original exclusion log/rules.



## Sampling procedure documented for the paper

The frozen sampling record implements the procedure reported in the manuscript:

1. Exclude research-recruitment posts, promotional content, empty post bodies, and exact-text duplicates.
2. Construct a **200-post naturalistic core** allocated proportionally across the 12 RRCP subreddits.
3. Within each subreddit, order eligible posts by year and document word count and select posts systematically using a seeded random start.
4. Construct a **50-post safety-enriched challenge set** using an independent high-recall lexical screen. The screen is a *sampling device only* and is not a gold safety label.
5. Enforce one sampled post per author across the full 250-post benchmark.
6. Mask URLs, email addresses, user handles, and subreddit references in the annotation/prediction text.

The workbook contains the frozen sample, sampling split/stratum, safety-screen rules, and audit counts. Annotators should not receive the researcher-only sampling metadata.


In [ ]:

# Colab install cell
!pip install -q -U pandas openpyxl


In [ ]:

from pathlib import Path
import json
import re
import zipfile

import numpy as np
import pandas as pd

RANDOM_SEED = 20260903
EXPECTED_N = 250
EXPECTED_CORE = 200
EXPECTED_CHALLENGE = 50

EXPECTED_CHALLENGE_STRATA = {
    "safety_candidate_direct": 20,
    "safety_candidate_indirect": 10,
    "safety_candidate_medical": 10,
    "safety_candidate_violence_abuse": 10,
}

EXPECTED_CORE_BY_SUBREDDIT = {
    "migraine": 44,
    "CrohnsDisease": 53,
    "backpain": 6,
    "Interstitialcystitis": 8,
    "fibromyalgia": 25,
    "ChronicPain": 28,
    "rheumatoid": 7,
    "Sciatica": 6,
    "ankylosingspondylitis": 6,
    "lupus": 10,
    "ChronicIllness": 4,
    "Thritis": 3,
}

OUTPUT_DIR = Path("sampling_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Seed:", RANDOM_SEED)


## Upload the researcher-only sampling workbook

In [ ]:

# In Colab, upload the sampling workbook when prompted.
# If you prefer Google Drive, replace WORKBOOK with a Drive path.

from google.colab import files

uploaded = files.upload()

xlsx_files = [name for name in uploaded if name.lower().endswith(".xlsx")]
if not xlsx_files:
    raise FileNotFoundError("Please upload reddit_250_sampling.xlsx")

if len(xlsx_files) > 1:
    print("Multiple XLSX files uploaded; using:", xlsx_files[0])

WORKBOOK = Path(xlsx_files[0])
print("Using:", WORKBOOK)


## Load the frozen sample and audit sheets

In [ ]:

required_sheets = {
    "README",
    "Master_Sample",
    "Sampling_Audit",
    "Safety_Screen_Rules",
}

book = pd.ExcelFile(WORKBOOK)
missing_sheets = required_sheets - set(book.sheet_names)
if missing_sheets:
    raise ValueError(f"Workbook is missing sheets: {sorted(missing_sheets)}")

master = pd.read_excel(WORKBOOK, sheet_name="Master_Sample")
audit_raw = pd.read_excel(WORKBOOK, sheet_name="Sampling_Audit", header=None)
rules_raw = pd.read_excel(WORKBOOK, sheet_name="Safety_Screen_Rules", header=None)

print("Rows in Master_Sample:", len(master))
print("Columns:", list(master.columns))
master.head(3)


## Validate the frozen 250-post benchmark

In [ ]:

required_columns = {
    "sample_id",
    "sampling_split",
    "sampling_stratum",
    "source_id",
    "source_author",
    "subreddit",
    "year",
    "title",
    "selftext",
    "document",
    "annotation_text",
    "document_word_count",
}

missing = required_columns - set(master.columns)
if missing:
    raise ValueError(f"Missing Master_Sample columns: {sorted(missing)}")

assert len(master) == EXPECTED_N, f"Expected {EXPECTED_N} rows, found {len(master)}"
assert master["sample_id"].nunique() == EXPECTED_N, "sample_id values are not unique"
assert master["source_id"].astype(str).nunique() == EXPECTED_N, "source_id values are not unique"
assert master["source_author"].astype(str).nunique() == EXPECTED_N, "Expected one sampled post per author"

split_counts = master["sampling_split"].value_counts().to_dict()
assert split_counts.get("core", 0) == EXPECTED_CORE, split_counts
assert split_counts.get("safety_challenge", 0) == EXPECTED_CHALLENGE, split_counts

challenge = master[master["sampling_split"].eq("safety_challenge")]
challenge_counts = challenge["sampling_stratum"].value_counts().to_dict()

for stratum, expected in EXPECTED_CHALLENGE_STRATA.items():
    actual = int(challenge_counts.get(stratum, 0))
    assert actual == expected, f"{stratum}: expected {expected}, found {actual}"

print("✓ 250 unique posts")
print("✓ 250 unique authors")
print("✓ 200 core + 50 safety challenge")
print("✓ Challenge strata = 20 / 10 / 10 / 10")


## Check text construction, masking, and duplicate protection

In [ ]:

def normalize_for_duplicate_check(text):
    text = "" if pd.isna(text) else str(text)
    text = text.lower()
    text = re.sub(r"https?://\S+|www\.\S+", "[url]", text)
    text = re.sub(r"\b[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}\b", "[email]", text)
    text = re.sub(r"(?<!\w)[uU]/[A-Za-z0-9_-]+", "[user]", text)
    text = re.sub(r"(?<!\w)@[A-Za-z0-9_]+", "[user]", text)
    text = re.sub(r"(?<!\w)[rR]/[A-Za-z0-9_]+", "[subreddit]", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def word_count(text):
    return len(re.findall(r"\b\w+\b", "" if pd.isna(text) else str(text)))

master = master.copy()
master["recomputed_word_count"] = master["document"].apply(word_count)
master["normalized_text_check"] = master["annotation_text"].apply(normalize_for_duplicate_check)

if master["normalized_text_check"].duplicated().any():
    dup = master.loc[
        master["normalized_text_check"].duplicated(keep=False),
        ["sample_id", "source_id"]
    ]
    raise ValueError(f"Duplicate normalized texts detected:\n{dup}")

wc_diff = (
    master["recomputed_word_count"].astype(int)
    - pd.to_numeric(master["document_word_count"], errors="coerce").fillna(-1).astype(int)
).abs()

print("Duplicate normalized texts:", int(master["normalized_text_check"].duplicated().sum()))
print("Rows where recomputed word count differs from stored count:", int((wc_diff != 0).sum()))
print("Rows where annotation_text differs from raw document due to masking:",
      int((master["annotation_text"].astype(str) != master["document"].astype(str)).sum()))


## Verify the naturalistic-core allocation

In [ ]:

core = master[master["sampling_split"].eq("core")].copy()

core_by_subreddit = core["subreddit"].value_counts().to_dict()

check_rows = []
for subreddit, expected in EXPECTED_CORE_BY_SUBREDDIT.items():
    actual = int(core_by_subreddit.get(subreddit, 0))
    check_rows.append({
        "subreddit": subreddit,
        "expected_n": expected,
        "actual_n": actual,
        "matches": actual == expected,
    })

core_check = pd.DataFrame(check_rows).sort_values("subreddit")
display(core_check)

if not core_check["matches"].all():
    raise ValueError("Core subreddit allocation does not match the frozen study sample.")

print("✓ Core allocation matches the study record")


## Show the safety-enriched challenge composition

In [ ]:

challenge_summary = (
    challenge.groupby(["sampling_stratum", "subreddit"])
    .size()
    .rename("n")
    .reset_index()
)

display(
    challenge["sampling_stratum"]
    .value_counts()
    .rename_axis("sampling_stratum")
    .reset_index(name="n")
)

display(challenge_summary)



## Export the prediction input

The prediction notebook intentionally receives only:

- `post_id`
- `post_text`

This prevents gold labels, subreddit information, sampling strata, authors, and source URLs from leaking into model inference.

`annotation_text` is used because it is the masked narrative text stored in the frozen sampling record.


In [ ]:

prediction_input = master[["sample_id", "annotation_text"]].rename(
    columns={
        "sample_id": "post_id",
        "annotation_text": "post_text",
    }
)

if prediction_input["post_id"].duplicated().any():
    raise ValueError("Duplicate post_id values in prediction input.")

if prediction_input["post_text"].isna().any():
    raise ValueError("Missing post_text values in prediction input.")

prediction_path = OUTPUT_DIR / "Reddit_250_Prediction_Input.csv"
prediction_input.to_csv(prediction_path, index=False)

print("Saved:", prediction_path)
print("Rows:", len(prediction_input))
prediction_input.head(3)



## Export a shareable sampling audit

The researcher-only workbook contains raw Reddit text, authors, and source URLs. Do **not** put that workbook in a public repository unless you have explicitly decided that redistribution is appropriate.

The following public audit contains only aggregate/methodological fields by default. It omits raw text, author names, URLs, and safety-screen match terms.

`source_id` is also omitted by default because it can make individual posts easier to retrieve. Set `INCLUDE_SOURCE_ID = True` only if you have decided that releasing source IDs is appropriate for your data-governance plan.


In [ ]:

INCLUDE_SOURCE_ID = False

manifest_columns = [
    "sample_id",
    "sampling_split",
    "sampling_stratum",
    "subreddit",
    "year",
    "document_word_count",
]

if INCLUDE_SOURCE_ID:
    manifest_columns.insert(1, "source_id")

public_manifest = master[manifest_columns].copy()

manifest_path = OUTPUT_DIR / "RRCP_250_Sampling_Manifest_NoText.csv"
public_manifest.to_csv(manifest_path, index=False)

method_summary = {
    "random_seed": RANDOM_SEED,
    "n_total": int(len(master)),
    "n_core": int((master["sampling_split"] == "core").sum()),
    "n_safety_challenge": int((master["sampling_split"] == "safety_challenge").sum()),
    "challenge_targets": EXPECTED_CHALLENGE_STRATA,
    "one_post_per_author": bool(master["source_author"].astype(str).nunique() == len(master)),
    "unique_source_ids": bool(master["source_id"].astype(str).nunique() == len(master)),
    "duplicate_normalized_texts": int(master["normalized_text_check"].duplicated().sum()),
    "prediction_input_columns": ["post_id", "post_text"],
    "public_manifest_includes_source_id": INCLUDE_SOURCE_ID,
}

summary_path = OUTPUT_DIR / "sampling_method_summary.json"
with summary_path.open("w", encoding="utf-8") as f:
    json.dump(method_summary, f, indent=2)

print("Saved:", manifest_path)
print("Saved:", summary_path)
display(public_manifest.head())



## Optional: inspect the sampling-only safety screen

These phrases were used only to create a high-recall candidate pool for the 50-post challenge subset. They are **not reference labels** and were not shown to the annotators.

This cell simply displays the rules already stored in the researcher-only workbook.


In [ ]:

display(rules_raw.dropna(how="all"))


## Package outputs for download

In [ ]:

zip_path = Path("InterLeaf_sampling_outputs.zip")

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for path in OUTPUT_DIR.iterdir():
        if path.is_file():
            z.write(path, arcname=path.name)

print("Created:", zip_path)

from google.colab import files
files.download(str(zip_path))



## Repository guidance

For the paper repository, a conservative release is:

1. `Interleaf_RRCP_Sampling_and_Preparation.ipynb`
2. `Interleaf_AI_Pipeline_Predictions_Reddit.ipynb`
3. `Interleaf_AI_Pipeline_Evaluation_Reddit.ipynb`
4. aggregate evaluation outputs that do not reproduce raw Reddit text, if you choose to share them.

The researcher-only `reddit_250_sampling.xlsx` contains raw source material and identifying platform metadata and should not automatically be treated as a public artifact.

A concise manuscript statement is:

> The code used for sample preparation, AI-pipeline inference, and evaluation is available at: [ANONYMIZED-REPOSITORY-LINK].

If you later decide to release gold annotations, saved model predictions, or aggregate evaluation tables, those are **derived evaluation materials** and can be named explicitly in the availability statement.
